# Bible Notely — YouTube no Google Colab

Lê os MP4 no **Drive montado** (`/content/drive/MyDrive/...`) e envia ao YouTube pela API. O arquivo **não passa pelo seu PC**.

Título: `IBQB - EBD|Culto - DD/MM/AAAA - {título do estudo}` · não listado · playlists EBD / culto manhã / culto noite.

## Uma vez
1. Google Cloud: YouTube Data API v3 + cliente OAuth **Desktop**.
2. Salve `client_secret.json` em `MyDrive/auto-ebd-youtube/`.
3. Copie `colab/youtube_lib.py` para essa mesma pasta (ou deixe o clone do repo no Drive).
4. Na máquina: **Ajustes → YouTube** (pasta) e `cli youtube-catalog` (gera `catalog.json`).
5. Depois do lote: `cli youtube-sync` (URLs no SQLite).

Cota: ~6 vídeos/dia. Token fica no Drive (`token.json`) e vale nas próximas sessões.

In [ ]:
!pip install --upgrade google-api-python-client google-auth-oauthlib google-auth-httplib2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import importlib.util
import shutil
import urllib.request

# Ajuste só estes caminhos do Drive montado
EBD_ROOT = Path('/content/drive/MyDrive/EBD')  # pasta das aulas ANO/Mês/Dia
META_FOLDER_ID = '1ejQ8Wz1wMylcuehWSQmyhg26fXiowVEb'
META_DIR = Path('/content/drive/MyDrive/auto-ebd-youtube')  # mesma pasta do ID acima
UPLOAD_SCOPE = ''  # ex.: '2025/09-Setembro' — vazio = o que estiver no catalog.json
MAX_UPLOADS = 6
LIB_URL = 'https://raw.githubusercontent.com/scalet/bible-notely/main/colab/youtube_lib.py'

META_DIR.mkdir(parents=True, exist_ok=True)

LIB_CANDIDATES = [
    META_DIR / 'youtube_lib.py',
    Path('/content/drive/MyDrive/DEV/auto-ebd/colab/youtube_lib.py'),
    Path('/content/drive/MyDrive/auto-ebd/colab/youtube_lib.py'),
]
lib_src = next((p for p in LIB_CANDIDATES if p.exists()), None)
if lib_src is None:
    dest = META_DIR / 'youtube_lib.py'
    print('Baixando youtube_lib.py do GitHub…')
    urllib.request.urlretrieve(LIB_URL, dest)
    lib_src = dest
runtime_lib = Path('/content/youtube_lib.py')
shutil.copy(lib_src, runtime_lib)
if lib_src != META_DIR / 'youtube_lib.py':
    shutil.copy(lib_src, META_DIR / 'youtube_lib.py')

spec = importlib.util.spec_from_file_location('youtube_lib', runtime_lib)
yt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(yt)
print('youtube_lib:', lib_src)
print('EBD_ROOT existe:', EBD_ROOT.is_dir(), EBD_ROOT)
print('META_DIR:', META_DIR)

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

SCOPES = [
    'https://www.googleapis.com/auth/youtube.upload',
    'https://www.googleapis.com/auth/youtube',
]

SECRET_PATH = META_DIR / 'client_secret.json'
TOKEN_PATH = META_DIR / 'token.json'
CATALOG_PATH = META_DIR / 'catalog.json'
STATE_PATH = META_DIR / 'state.json'
if not CATALOG_PATH.exists() and (EBD_ROOT / 'auto-ebd-youtube' / 'catalog.json').exists():
    CATALOG_PATH = EBD_ROOT / 'auto-ebd-youtube' / 'catalog.json'

creds = yt.load_colab_credentials(SECRET_PATH, TOKEN_PATH, SCOPES)
youtube = build('youtube', 'v3', credentials=creds)
print('YouTube autenticado. Token em', TOKEN_PATH)

In [ ]:
catalog = yt.read_json(CATALOG_PATH) or {'items': []}
state = yt.load_state(yt.read_json(STATE_PATH))
catalog_by_id = yt.catalog_index(catalog)
print('Catálogo:', CATALOG_PATH, 'itens=', len(catalog.get('items') or []))
print('Já no state:', len(state.get('uploads') or {}))

if not EBD_ROOT.is_dir():
    raise FileNotFoundError(
        f'Pasta EBD não encontrada: {EBD_ROOT}. Ajuste EBD_ROOT para o caminho em MyDrive.'
    )

videos = yt.list_mounted_mp4s(EBD_ROOT, UPLOAD_SCOPE)
print('MP4 no Drive montado (escopo):', len(videos))

rows = [yt.merge_video_row(v, catalog_by_id, state) for v in videos]
if catalog.get('items') and not videos:
    print('Nenhum MP4 na árvore montada; tentando caminhos do catálogo…')
    rows = []
    for item in catalog['items']:
        local = yt.resolve_local_video(EBD_ROOT, item.get('drive_path') or '', item.get('video_filename') or '')
        video = {
            'id': item.get('drive_file_id') or '',
            'name': item.get('video_filename') or '',
            'path': item.get('drive_path') or '',
            'size': local.stat().st_size if local.is_file() else 0,
            'local_path': str(local),
            'lecture_date': item.get('lecture_date') or '',
        }
        rows.append(yt.merge_video_row(video, catalog_by_id, state))

rows.sort(key=lambda r: (r.get('lecture_date') or '', r.get('video_filename') or ''), reverse=True)
print('\nPreview (até 40):')
for r in rows[:40]:
    print(
        r['drive_path'], '|', r['video_filename'], '|', r['playlist'],
        '|', r['youtube_title'] or '(sem título)', '|', r['skip_reason'] or 'OK',
    )
ready = [r for r in rows if not r['skip_reason']]
print(f'\nProntos para enviar: {len(ready)} (MAX_UPLOADS={MAX_UPLOADS})')

In [ ]:
state = yt.ensure_playlists(youtube, state)
yt.write_json(STATE_PATH, state)
print('Playlists:', state.get('playlists'))

sent = skipped = failed = 0
for row in rows:
    if sent >= MAX_UPLOADS:
        print('Teto do dia (MAX_UPLOADS) atingido.')
        break
    if row['skip_reason']:
        if row['skip_reason'].startswith('arquivo não encontrado'):
            print('PULADO (não achei o vídeo):', row['skip_reason'])
        skipped += 1
        continue

    local = Path(row['local_path']) if row.get('local_path') else yt.resolve_local_video(
        EBD_ROOT, row['drive_path'], row['video_filename']
    )
    print('Enviando', local, '→', row['youtube_title'], flush=True)
    try:
        video_id = yt.upload_video_file(youtube, local, row)
        yt.add_to_playlist(youtube, state['playlists'][row['playlist']], video_id)
        yt.record_upload(
            state,
            drive_file_id=row['drive_file_id'],
            youtube_id=video_id,
            title=row['youtube_title'],
            drive_path=row['drive_path'],
            playlist=row['playlist'],
        )
        yt.write_json(STATE_PATH, state)
        print('  OK', yt.youtube_watch_url(video_id), flush=True)
        sent += 1
    except FileNotFoundError as exc:
        print('  ERRO arquivo:', exc, flush=True)
        failed += 1
    except HttpError as exc:
        print('  ERRO YouTube HTTP:', exc, flush=True)
        failed += 1
        if yt.is_youtube_quota_error(exc):
            print('Cota da YouTube Data API esgotada. Pare e retome amanhã.', flush=True)
            break
    except Exception as exc:
        print('  ERRO:', type(exc).__name__, exc, flush=True)
        failed += 1
        if yt.is_youtube_quota_error(exc):
            print('Cota da YouTube Data API esgotada. Pare e retome amanhã.', flush=True)
            break

print(f'Resumo: enviados={sent} pulados={skipped} falhas={failed}')
print('State salvo em', STATE_PATH)
print('Na sua máquina (só metadado): PYTHONPATH=$PWD .venv/bin/python -m backend.app.cli youtube-sync')